In this Colab, you'll go from a raw pharmaceutical PDF blob to page-level classification, logical document grouping, chunking, embedding with metadata, and filtered retrieval.

By the end, your system will:

* Classify pharmaceutical documents like Certificates of Quality, Packaging Specifications, and BSE/TSE Declarations
* Retrieve only relevant chunks using smart filters like `doc_type`
* Show improved performance in noisy blob-style files containing multiple document types
* Be fully modular for use in production document intelligence pipelines

# Setup

In [2]:
# ============================================
# STEP 1: Install Required Packages
# ============================================
#
# WHAT WE'RE DOING:
# Installing LlamaIndex (RAG framework), HuggingFace embeddings,
# PyPDF2 (PDF text extraction), and Google Generative AI (Gemini LLM).
#
# WHY THIS MATTERS:
# This end-to-end pipeline combines all the tools from previous notebooks:
# PDF reading, LLM classification, embedding, and filtered retrieval.
#
# WHAT YOU'LL SEE:
# Package installation output (may take 1-2 minutes).
# ============================================

!pip install -q llama-index llama-index-readers-file llama-index-embeddings-huggingface
!pip install -q transformers sentence-transformers PyPDF2


In [4]:
# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.5/444.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 35.1 MB/s eta 0:00:00


# Step 2: Load the Pharmaceutical Blob PDF and Extract Pages

In [5]:
# ============================================
# STEP 2: Load the Pharmaceutical Blob PDF
# ============================================
#
# WHAT WE'RE DOING:
# Reading the 10-page pharmaceutical blob PDF and extracting
# text from each page using PyPDF2.
#
# WHY THIS MATTERS:
# This blob contains 7 different document types (Cover Letter,
# Certificates of Quality, Packaging Spec, BSE/TSE Declaration,
# Material Description, Supplier Qualification, Chain of Custody).
# We need to split and classify them automatically.
#
# WHAT YOU'LL SEE:
# A list of dictionaries, one per page, each with page_num and text.
# ============================================

from PyPDF2 import PdfReader

reader = PdfReader("/content/pharma-blob-sample.pdf")
pages = [page.extract_text() for page in reader.pages]
doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]
doc_pages

[{'page_num': 0,
  'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescription Part Number Operating Temperature\nHigh Flow Kit F, Modified, AKTA ready 29477427 +2°C to +40°C\nHigh Flow Gradient C, Modified, AKTA ready 29184612 +2°C to +4

# Step 3: Use LLM to Classify Document Type and Boundaries

In [8]:
from llama_cpp import Llama
import os

model_path = "/content/mistral.gguf"  # change this to your model path
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

--2026-07-12 17:23:20--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.165.160.12, 3.165.160.11, 3.165.160.59, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.12|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65778ac662d3ac1817cc9201/865f5e4682dddb29c2e20270b2471a7590c83a414bbf1d72cf4c08fdff2eeca4?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.2.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.2.Q4_K_M.gguf%22%3B&Expires=1783880600&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU3NzhhYzY2MmQzYWMxODE3Y2M5MjAxLzg2NWY1ZTQ2ODJkZGRiMjljMmUyMDI3MGIyNDcxYTc1OTBjODNhNDE0YmJmMWQ3MmNmNGMwOGZkZmYyZWVjYTRcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRp

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model downloaded to /content/mistral.gguf
Model file exists. Size: 4166.07 MB


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
llm_load_tensors: ggml ctx size =    0.27 MiB
llm_load_tensors: offloading 1 repeating layers to GPU
llm_load_tensors: offloaded 1/33 layers to GPU
llm_load_tensors:        CPU buffer size =  4165.37 MiB
llm_load_tensors:      CUDA0 buffer size =   132.50 MiB
.................................................................................................
llama_new_context_with_model: n_ctx      = 2048
llama_new_context_with_model: n_batch    = 512
llama_new_context_with_model: n_ubatch   = 512
llama_new_context_with_model: flash_attn = 0
llama_new_context_with_model: freq_base  = 1000000.0
llama_new_context_with_model: freq_scale = 1
llama_kv_cache_init:  CUDA_Host KV buffer size =   248.00 MiB
llama_kv_cache_init:      CUDA0 KV buffer size =     8.00 MiB
llama_new_context_with_model: KV self size  =  256.00

Model loaded successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | 
Model metadata: {'tokenizer.chat_template': "{{ bos_token }}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token}}{% else %}{{ raise_exception('Only user and assistant roles are supported!') }}{% endif %}{% endfor %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': 

In [9]:
# ============================================
# STEP 3: Set Up Local Mistral LLM
# ============================================
#
# WHAT WE'RE DOING:
# Creating a helper function that sends prompts to Google's
# Gemini 2.0 Flash model. We'll use this for classification
# and boundary detection.
#
# WHY THIS MATTERS:
# The LLM is the brain of our pipeline — it decides what type
# each page is and whether consecutive pages belong together.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================

def gemini_model(prompt):
    """
    Keeps the same function name used in the notebook,
    but now calls the local open-source Mistral model
    instead of Gemini.
    """

    formatted_prompt = f"""
[INST]
{prompt}
[/INST]
"""

    response = llm(
        formatted_prompt,
        max_tokens=300,
        temperature=0.0,
        stop=["</s>", "[/INST]"]
    )

    return response["choices"][0]["text"].strip()

In [10]:
# ============================================
# STEP 4: Define Document Boundary Detection
# ============================================
#
# WHAT WE'RE DOING:
# Creating a function that asks the LLM whether two consecutive
# pages belong to the same pharmaceutical document.
#
# WHY THIS MATTERS:
# Some documents span multiple pages (e.g., a 2-page Packaging
# Specification or 2-page Supplier Qualification Record).
# We need to detect these boundaries accurately.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================

def is_same_document(prev_text, curr_text, doc_type=None):
    prompt = f"""
    You are checking whether two consecutive pages from a pharmaceutical
    document package belong to the SAME individual document.

    A NEW document starts when the page has:
    - A different document title or heading (e.g., "Certificate of Quality"
      vs "Packaging Specification" vs "Material Description Sheet")
    - A completely different topic or subject matter
    - Its own header with a new document number or reference

    Pages belong to the SAME document when:
    - The second page says "continued" or "page 2 of 2"
    - The content directly continues the previous page's discussion
    - They share the same document number or title

    Previous page type: {doc_type or 'unknown'}

    Previous Page:
    {prev_text}

    Current Page:
    {curr_text}

    Do these two pages belong to the SAME document? Answer ONLY 'Yes' or 'No'.
    """
    response = gemini_model(prompt)
    return response.strip().lower().startswith("yes")

In [11]:
# ============================================
# STEP 5: Define Document Type Classification
# ============================================
#
# WHAT WE'RE DOING:
# Creating a function that classifies a page into one of
# 7 pharmaceutical document types (plus "Other").
#
# WHY THIS MATTERS:
# Each document type needs different extraction logic.
# A Certificate of Quality has test results and lot numbers,
# while a BSE/TSE Declaration has compliance dates.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================

# Valid document type labels used throughout this notebook
VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def clean_doc_type(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    # Check if the cleaned response contains a valid label
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title

def classify_document_type(text):
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:

    - Cover Letter: A formal letter (often starting with "To Whom It
      May Concern") discussing product information or storage conditions.
    - Certificate of Quality: Contains lot numbers, manufacture dates,
      expiration dates, and test results (autoclave, gamma irradiation).
    - Packaging Specification: Describes packaging components, materials,
      part numbers, and configuration change history.
    - BSE/TSE Declaration: A declaration about animal-origin materials
      and transmissible spongiform encephalopathy compliance.
    - Material Description: Lists materials of construction, sterilization
      compatibility, and physical properties of a product.
    - Supplier Qualification: Contains supplier audit history,
      certifications (ISO 9001, ISO 13485), and approved product lists.
    - Chain of Custody: Lists manufactured assemblies, traceability
      information, and the manufacturing-to-shipment flow.
    - Other: Use ONLY if the content does not match any of the above.

    Page Content:
           {text}

           Respond with ONLY the document type name. No explanation.
           """
    response = gemini_model(prompt)
    return clean_doc_type(response)

In [12]:
# ============================================
# STEP 6: Run the Classification Pipeline
# ============================================
#
# WHAT WE'RE DOING:
# Looping through all 10 pages, classifying each one and
# detecting document boundaries. This builds our metadata_store.
#
# WHY THIS MATTERS:
# This is the core of the blob processing pipeline — it turns
# a flat list of pages into structured, labeled metadata.
#
# WHAT YOU'LL SEE:
# A list of 10 entries, each with page number, text, doc_type,
# and page_in_doc (0 = first page of that document).
# Multi-page docs like Packaging Spec (pages 4-5) will show
# page_in_doc: 0 then 1.
# ============================================

metadata_store = []
current_doc_type = None
doc_counter = 0
page_counter = 0
for i, page in enumerate(doc_pages):
    same = False
    if i == 0:
        current_doc_type = classify_document_type(page["text"])
    else:
        prev_text = doc_pages[i - 1]["text"]
        same = is_same_document(prev_text, page["text"], current_doc_type)
        if not same:
            doc_counter += 1
            current_doc_type = classify_document_type(page["text"])
            page_counter = 0
        else:
            page_counter += 1

    metadata_store.append({
        "page": i,
        "text": page["text"],
        "doc_type": current_doc_type,
        "page_in_doc": page_counter
    })

metadata_store


llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =       0.22 ms /     5 runs   (    0.04 ms per token, 22321.43 tokens per second)
llama_print_timings: prompt eval time =    4305.66 ms /   705 tokens (    6.11 ms per token,   163.74 tokens per second)
llama_print_timings:        eval time =    2475.44 ms /     4 runs   (  618.86 ms per token,     1.62 tokens per second)
llama_print_timings:       total time =    6791.02 ms /   709 tokens
Llama.generate: 11 prefix-match hit, remaining 863 prompt tokens to eval

llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =       4.37 ms /    85 runs   (    0.05 ms per token, 19441.90 tokens per second)
llama_print_timings: prompt eval time =    2913.24 ms /   863 tokens (    3.38 ms per token,   296.23 tokens per second)
llama_print_timings:        eval time =   54759.71 ms /    84 runs   (  651.90 ms per token,     1.53 tokens per second)
llama_print_timings:  

[{'page': 0,
  'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescription Part Number Operating Temperature\nHigh Flow Kit F, Modified, AKTA ready 29477427 +2°C to +40°C\nHigh Flow Gradient C, Modified, AKTA ready 29184612 +2°C to +40°C\

# Step 4: Group Pages into Logical Documents

In [13]:
# ============================================
# STEP 7: Group Pages into Logical Documents
# ============================================
#
# WHAT WE'RE DOING:
# Combining consecutive pages that belong to the same document
# into a single logical document with merged text.
#
# WHY THIS MATTERS:
# The Packaging Specification (pages 4-5) and Supplier
# Qualification Record (pages 8-9) each span 2 pages.
# We need to merge them so retrieval sees the full document.
#
# WHAT YOU'LL SEE:
# No printed output, but logical_docs will contain ~8 documents
# (10 pages grouped into 8 logical documents).
# ============================================

logical_docs = []
current_doc = {"text": "", "doc_type": None, "page_start": 0}

for page in metadata_store:
    if page["page_in_doc"] == 0 and current_doc["text"]:
        current_doc["page_end"] = page["page"] - 1
        logical_docs.append(current_doc)
        current_doc = {"text": "", "doc_type": None, "page_start": page["page"]}

    current_doc["text"] += "\n\n" + page["text"]
    current_doc["doc_type"] = page["doc_type"]

# Append last document
current_doc["page_end"] = metadata_store[-1]["page"]
logical_docs.append(current_doc)

# Step 5: Chunk Each Logical Document with Metadata

In [14]:
# ============================================
# STEP 8: Chunk Each Document with Metadata
# ============================================
#
# WHAT WE'RE DOING:
# Splitting each logical document into 500-word chunks and
# attaching metadata (doc_type, page range, source file).
#
# WHY THIS MATTERS:
# Chunks are what actually get embedded and stored in the
# vector database. The metadata lets us filter retrieval
# to only the right document type.
#
# WHAT YOU'LL SEE:
# No printed output, but chunked_documents will contain
# LlamaIndex Document objects ready for embedding.
# ============================================

from llama_index.core.schema import Document

def chunk_text(text, chunk_size=500):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

chunked_documents = []

for idx, doc in enumerate(logical_docs):
    chunks = chunk_text(doc["text"])
    for chunk_idx, chunk in enumerate(chunks):
        chunked_documents.append(
            Document(
                text=chunk,
                metadata={
                    "doc_type": doc["doc_type"],
                    "chunk_index": chunk_idx,
                    "page_start": doc["page_start"],
                    "page_end": doc["page_end"],
                    "source_file": "pharma-blob-sample.pdf"
                }
            )
        )

# Step 6: Embed and Store in Vector DB with Metadata

In [15]:
# ============================================
# STEP 9: Embed Chunks and Build Vector Index
# ============================================
#
# WHAT WE'RE DOING:
# Converting each chunk into a numerical vector (embedding)
# using the BGE-small embedding model, then storing them in
# an in-memory vector index for fast similarity search.
#
# WHY THIS MATTERS:
# This is what makes RAG work — when a user asks a question,
# we find the most similar chunks by comparing embeddings.
#
# WHAT YOU'LL SEE:
# A progress bar as chunks are embedded (may take 30-60 seconds).
# ============================================

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents(chunked_documents, embed_model=embed_model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Step 7: Predict Query Routing and Retrieve with Metadata Filter

In [16]:
# ============================================
# STEP 10: Define Query Routing Function
# ============================================
#
# WHAT WE'RE DOING:
# Creating a function that predicts which document type is
# most relevant to a user's query. The LLM acts as a router,
# directing each query to the right document type.
#
# WHY THIS MATTERS:
# Without routing, a query about BSE/TSE compliance might
# retrieve text from a Cover Letter. Routing ensures we
# search only within the relevant document type.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================

def predict_doc_type_for_query(query):
    prompt = f"""
  You are an intelligent assistant that routes user queries to the most relevant pharmaceutical document.

  User query: "{query}"

  Which document type is most likely to contain the answer?
  Choose ONLY ONE from: Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration,
Material Description, Supplier Qualification, Chain of Custody, Other.

  Respond with ONLY the document type name. No explanation.
  """

    response = gemini_model(prompt)
    return clean_doc_type(response)

In [17]:
# ============================================
# STEP 11: Route a Query and Retrieve Filtered Results
# ============================================
#
# WHAT WE'RE DOING:
# Testing the full pipeline with a pharmaceutical query.
# The LLM predicts the document type, then we retrieve
# only chunks from that type using metadata filtering.
#
# WHY THIS MATTERS:
# This is the payoff — instead of searching all 10 pages,
# we narrow down to just the relevant document type first.
#
# WHAT YOU'LL SEE:
# The predicted document type (e.g., "Packaging Specification"),
# followed by matching chunks with page ranges.
# ============================================

from llama_index.core.vector_stores import MetadataFilters, FilterOperator, MetadataFilter

# Get doc_type filter
user_query = "Were there any packaging configuration changes?"
predicted_doc_type = predict_doc_type_for_query(user_query)
print(f"Predicted document type: {predicted_doc_type}")

# Retrieve
retriever = index.as_retriever(
    filters=MetadataFilters(
        filters=[
            MetadataFilter(key="doc_type", value=predicted_doc_type, operator=FilterOperator.EQ)
        ]
    )
)
results = retriever.retrieve(user_query)

for res in results:
    print(f"[{res.metadata['doc_type']} - p{res.metadata['page_start']}–{res.metadata['page_end']}]")
    print(res.text[:200])
    print("-" * 50)

Llama.generate: 8 prefix-match hit, remaining 118 prompt tokens to eval

llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =       0.29 ms /     6 runs   (    0.05 ms per token, 20547.95 tokens per second)
llama_print_timings: prompt eval time =    1415.10 ms /   118 tokens (   11.99 ms per token,    83.39 tokens per second)
llama_print_timings:        eval time =    3373.28 ms /     5 runs   (  674.66 ms per token,     1.48 tokens per second)
llama_print_timings:       total time =    4793.71 ms /   123 tokens


Predicted document type: Packaging Specification
[Packaging Specification - p3–4]
Packaging Component Specification Document Number: PKG-SPEC-2023-0847 Revision: C Effective Date: 2023-11-08 Component: AKTA ready Flow Kit Packaging Assembly Drawing Reference: DWG-29477427-PKG-R3 1.
--------------------------------------------------


In [18]:
# ============================================
# STEP 12: Extract the Answer Using the LLM
# ============================================
#
# WHAT WE'RE DOING:
# Taking the top retrieved chunk and asking Gemini to extract
# a specific answer to the user's question.
#
# WHY THIS MATTERS:
# Retrieval alone gives us relevant text. This final step
# uses the LLM to produce a clear, concise answer — the
# "generation" part of Retrieval-Augmented Generation.
#
# WHAT YOU'LL SEE:
# A direct answer to the query extracted from the retrieved text.
# ============================================

if results:
    # Take the text from the first (most relevant) result
    retrieved_text = results[0].text

    # Ask the model for a specific answer
    prompt = f"""
    Based on the following text, please answer the user's question.

    Text:
    {retrieved_text}

    User Question: {user_query}

    Provide a clear, specific answer based only on the text provided.
    """

    answer = gemini_model(prompt)

    print(f"Query: '{user_query}'")
    print(f"Answer: {answer}")
else:
    print("Could not find an answer as no relevant documents were retrieved.")

Llama.generate: 8 prefix-match hit, remaining 619 prompt tokens to eval

llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =       3.71 ms /    75 runs   (    0.05 ms per token, 20237.45 tokens per second)
llama_print_timings: prompt eval time =    3152.23 ms /   619 tokens (    5.09 ms per token,   196.37 tokens per second)
llama_print_timings:        eval time =   47101.48 ms /    74 runs   (  636.51 ms per token,     1.57 tokens per second)
llama_print_timings:       total time =   50312.51 ms /   693 tokens


Query: 'Were there any packaging configuration changes?'
Answer: Yes, there have been packaging configuration changes based on the information provided in the text. Specifically, the blister material was changed from PVC to PETG (as indicated in ECN-2023-0847 and reflected in the updated drawing DWG-29477427-PKG-R3).


# Step 8: Test All 6 Extraction Queries

In [19]:
# ============================================
# STEP 13: Test All 6 Pharmaceutical Extraction Queries
# ============================================
#
# WHAT WE'RE DOING:
# Running all 6 extraction targets through the full pipeline
# to verify that routing, retrieval, and answer extraction
# work correctly for each document type.
#
# WHY THIS MATTERS:
# This is the ultimate test of our end-to-end system. Each
# query should route to the correct document type and extract
# a meaningful answer from the pharmaceutical blob.
#
# WHAT YOU'LL SEE:
# For each query: the predicted document type, the retrieved
# page range, and the extracted answer.
# ============================================

test_queries = [
    "What is the material description for this product?",
    "What are the part numbers listed in this document?",
    "Were there any packaging configuration changes?",
    "What are the BSE/TSE compliance dates?",
    "Are there any documents dated over 5 years ago?",
    "What supplier information is available?",
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"QUERY {i}: {query}")
    print(f"{'='*60}")

    # Route the query
    predicted_type = predict_doc_type_for_query(query)
    print(f"Routed to: {predicted_type}")

    # Retrieve with metadata filter
    retriever = index.as_retriever(
        filters=MetadataFilters(
            filters=[
                MetadataFilter(key="doc_type", value=predicted_type, operator=FilterOperator.EQ)
            ]
        )
    )
    results = retriever.retrieve(query)

    if results:
        print(f"Retrieved {len(results)} chunk(s) from pages {results[0].metadata['page_start']}-{results[0].metadata['page_end']}")

        # Extract answer
        prompt = f"""
        Based on the following text, please answer the user's question.

        Text:
        {results[0].text}

        User Question: {query}

        Provide a clear, specific answer based only on the text provided.
        """
        answer = gemini_model(prompt)
        print(f"Answer: {answer.strip()}")
    else:
        print("No matching documents found for this query.")


QUERY 1: What is the material description for this product?


Llama.generate: 8 prefix-match hit, remaining 119 prompt tokens to eval

llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =       0.19 ms /     4 runs   (    0.05 ms per token, 21390.37 tokens per second)
llama_print_timings: prompt eval time =    1255.92 ms /   119 tokens (   10.55 ms per token,    94.75 tokens per second)
llama_print_timings:        eval time =    1746.99 ms /     3 runs   (  582.33 ms per token,     1.72 tokens per second)
llama_print_timings:       total time =    3005.20 ms /   122 tokens
Llama.generate: 8 prefix-match hit, remaining 428 prompt tokens to eval


Routed to: Material Description
Retrieved 1 chunk(s) from pages 6-6



llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =      11.51 ms /   227 runs   (    0.05 ms per token, 19728.84 tokens per second)
llama_print_timings: prompt eval time =    1490.71 ms /   428 tokens (    3.48 ms per token,   287.11 tokens per second)
llama_print_timings:        eval time =  142223.06 ms /   226 runs   (  629.31 ms per token,     1.59 tokens per second)
llama_print_timings:       total time =  143922.27 ms /   654 tokens
Llama.generate: 8 prefix-match hit, remaining 120 prompt tokens to eval


Answer: The AKTA ready Gradient Flow Section is a pre-assembled flow path component designed for use with AKTA chromatography systems. The flow section includes inlet manifolds, pump tubing, and gradient mixing chambers. The housing is made of Polypropylene (PP), the gaskets are made of EPDM Rubber, the pump tubing and inlet fittings are made of Platinum-Cured Silicone, and the mixing chamber is made of Borosilicate Glass. The product is compatible with autoclave sterilization at 121°C for 15 minutes for the tubing, and gamma irradiation for the fittings. The dimensions of the product are 245 x 120 x 85 mm, it weighs 380 g when assembled, and it has an operating temperature range of +2°C to +40°C and an operating pressure of up to 5 MPa. The shelf life is 2 years from the date of manufacture.

QUERY 2: What are the part numbers listed in this document?


...

Routed to: Other
No matching documents found for this query.

QUERY 6: What supplier information is available?



llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =       0.26 ms /     6 runs   (    0.04 ms per token, 22813.69 tokens per second)
llama_print_timings: prompt eval time =    1082.35 ms /    90 tokens (   12.03 ms per token,    83.15 tokens per second)
llama_print_timings:        eval time =    2867.28 ms /     5 runs   (  573.46 ms per token,     1.74 tokens per second)
llama_print_timings:       total time =    3953.20 ms /    95 tokens
Llama.generate: 8 prefix-match hit, remaining 1055 prompt tokens to eval


Routed to: Supplier Qualification
Retrieved 1 chunk(s) from pages 7-9



llama_print_timings:        load time =    2991.28 ms
llama_print_timings:      sample time =      15.43 ms /   300 runs   (    0.05 ms per token, 19437.61 tokens per second)
llama_print_timings: prompt eval time =   17038.40 ms /  1055 tokens (   16.15 ms per token,    61.92 tokens per second)
llama_print_timings:        eval time =  193956.71 ms /   299 runs   (  648.68 ms per token,     1.54 tokens per second)
llama_print_timings:       total time =  211321.90 ms /  1354 tokens


Answer: The text provides the following information about the supplier Cytiva Sweden AB (SUP-2019-0847):

1. Supplier Name: Cytiva Sweden AB
2. Supplier Address: Bjorkgatan 30, SE-751 84 Uppsala, Sweden
3. Supplier Code: SUP-2019-0847
4. Qualification Status: Approved
5. Last On-Site Audit: 20 November 2023 with an approved result
6. Next Scheduled Audit: November 2025
7. Certifications: ISO 9001:2015 and ISO 13485:2016, FDA Establishment number 3007488211, and Active EU Authorized Rep
8. Approved Products: Several part numbers for various types of kits and sections
9. Supply Chain Information: Manufacturing sites in Eysins, Switzerland, and Uppsala, Sweden, and a warehouse in Marlborough, MA, USA. Sub-suppliers include DuPont, Wacker Chemie AG, Sabic Innovative Polypropylene resin, and Schott AG.
10. Quality Agreement: Annual audit rights, CAPA notification within 48 hours, and
